# Radiant RAG MCP Server — Test Notebook

Tests all ten MCP tools exposed by `radiant-rag-mcp` against a live server instance.

| | |
|---|---|
| **Transport** | Streamable HTTP (server in background thread, client on localhost) |
| **Backend** | ChromaDB (default — no external services) |
| **LLM** | Ollama Cloud `gemma4:31b-cloud` via `https://ollama.com/v1` |
| **LLM tools** | Cells marked `[LLM]` require `RADIANT_OLLAMA_API_KEY` — skipped automatically if absent |

**Before running:** create an API key at https://ollama.com/settings/keys and add it to
Colab Secrets (key icon in the left sidebar) as `OLLAMA_API_KEY`.

---

## Sections
1. Install
2. Import check
3. Configuration
4. Helpers
5. Server startup
6. Connection verification
7. Pre-ingestion state
8. Document ingestion
9. Search (no LLM)
10. Query `[LLM]`
11. Multi-turn conversation `[LLM]`
12. Maintenance
13. Summary


## 1  Install


In [1]:
import subprocess, sys

ACTIVE_BRANCH = 'mcp'
CONFIG_PATH   = '/content/config.yaml'

# ── Install ───────────────────────────────────────────────────────────────────
!pip install -q "radiant-rag-mcp[chroma] @ git+https://github.com/dshipley71/radiant-rag-mcp.git"
!pip install -q --prefer-binary nest_asyncio httpx "fastmcp>=3.0"
!wget -q "https://raw.githubusercontent.com/dshipley71/radiant-rag-mcp/config.yaml" -O {CONFIG_PATH}

# ── Pre-cache model weights ───────────────────────────────────────────────────
# Both models are loaded by the server at startup. Caching them here means
# startup is instantaneous instead of waiting for ~175 MB of downloads.
!python3 -c "from sentence_transformers import SentenceTransformer; \
    SentenceTransformer('sentence-transformers/all-MiniLM-L12-v2'); \
    print('sentence-transformers  ✓')"

!python3 -c "from sentence_transformers import CrossEncoder; \
    CrossEncoder('cross-encoder/ms-marco-MiniLM-L12-v2'); \
    print('cross-encoder          ✓')"

print('Install complete  ✓')


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 81.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 63.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 120.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 707.3/707.3 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.

## 2  Import check


In [2]:
import sys

_missing = []

# Check the MCP package itself
try:
    import radiant_rag_mcp
    print(f'  ✓  radiant_rag_mcp  ({radiant_rag_mcp.__file__})')
except ImportError:
    print('  ✗  radiant_rag_mcp  — re-run the install cell')
    _missing.append('radiant-rag-mcp (re-run install cell)')

# Check runtime dependencies
checks = [
    ('fastmcp',               'fastmcp'),
    ('nest_asyncio',          'nest_asyncio'),
    ('chromadb',              'chromadb'),
    ('sentence_transformers', 'sentence-transformers'),
    ('openai',                'openai'),
    ('yaml',                  'pyyaml'),
    ('httpx',                 'httpx'),
]

for module, pkg in checks:
    try:
        __import__(module)
        print(f'  ✓  {module}')
    except ImportError:
        print(f'  ✗  {module}  →  pip install --prefer-binary {pkg}')
        _missing.append(pkg)

print()
if _missing:
    print(f'[ACTION] Missing: {", ".join(_missing)}')
    print('         Fix the install cell and re-run before continuing.')
else:
    print('All imports resolved  ✓  — safe to continue')


  ✓  radiant_rag_mcp  (/usr/local/lib/python3.12/dist-packages/radiant_rag_mcp/__init__.py)
  ✓  fastmcp
  ✓  nest_asyncio
  ✓  chromadb
  ✓  sentence_transformers
  ✓  openai
  ✓  yaml
  ✓  httpx

All imports resolved  ✓  — safe to continue


## 3  Configuration

Set `RADIANT_OLLAMA_API_KEY` to enable LLM tools.
Get a key from https://ollama.com/settings/keys


In [3]:
import os
from google.colab import userdata

# ── Ollama Cloud ──────────────────────────────────────────────────────────────
# Base URL: HTTPS required, /v1 required (OpenAI-compatible API path).
# For local Ollama: 'http://localhost:11434/v1'
os.environ['RADIANT_OLLAMA_BASE_URL'] = 'https://ollama.com/v1'
os.environ['RADIANT_OLLAMA_API_KEY']  = userdata.get('OLLAMA_API_KEY')

# ── Transport ─────────────────────────────────────────────────────────────────
# HTTP transport: server and client coexist in the same Colab process.
os.environ['RADIANT_TRANSPORT'] = 'http'
os.environ['RADIANT_HOST']      = '127.0.0.1'
os.environ['RADIANT_PORT']      = '8765'

# ── Storage ───────────────────────────────────────────────────────────────────
os.environ['RADIANT_STORAGE_BACKEND'] = 'chroma'

# ── Config file ───────────────────────────────────────────────────────────────
os.environ['RADIANT_CONFIG_PATH'] = CONFIG_PATH

# ── Performance tuning ────────────────────────────────────────────────────────
os.environ['RADIANT_PIPELINE_USE_CRITIC']              = 'false'
os.environ['RADIANT_CITATION_ENABLED']                 = 'true'
os.environ['RADIANT_LLM_BACKEND_TIMEOUT']              = '30'
os.environ['RADIANT_LLM_BACKEND_MAX_RETRIES']          = '0'
os.environ['RADIANT_RERANKING_BACKEND_DEVICE']         = 'cuda' # GPU reranking
os.environ['RADIANT_EMBEDDING_BACKEND_DEVICE']         = 'cuda' # GPU embeddings
os.environ['RADIANT_CHUNKING_USE_LLM_CHUNKING']        = 'false'
os.environ['RADIANT_RETRIEVAL_DENSE_TOP_K']            = '5'
os.environ['RADIANT_RETRIEVAL_BM25_TOP_K']             = '5'

# ── Ollama Server ─────────────────────────────────────────────────────────────
SERVER_URL  = f"http://{os.environ['RADIANT_HOST']}:{os.environ['RADIANT_PORT']}/mcp"
HAS_LLM_KEY = bool(os.environ.get('RADIANT_OLLAMA_API_KEY'))

print(f'Branch        : {ACTIVE_BRANCH}')
print(f'Ollama URL    : {os.environ["RADIANT_OLLAMA_BASE_URL"]}')
print(f'LLM key set   : {HAS_LLM_KEY}')
print(f'Server URL    : {SERVER_URL}')
print(f'Backend       : {os.environ["RADIANT_STORAGE_BACKEND"]}')
print(f'Config        : {CONFIG_PATH}')

if not HAS_LLM_KEY:
    print()
    print('[NOTE] No API key — LLM cells will be skipped.')
    print('       Get a key at https://ollama.com/settings/keys')


Branch        : mcp
Ollama URL    : https://ollama.com/v1
LLM key set   : True
Server URL    : http://127.0.0.1:8765/mcp
Backend       : chroma
Config        : /content/config.yaml


## 4  Helpers


In [4]:
import asyncio
import json
import logging
import threading
import time
import textwrap
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()

import httpx as _httpx
from fastmcp import Client
from fastmcp.exceptions import ToolError


async def _call(tool: str, args: dict = None):
    """Call one MCP tool and return the parsed result.
    ToolErrors are caught and returned as {status: error, message: ...}
    rather than propagating — keeps cells running even when a tool fails.
    """
    try:
        async with Client(SERVER_URL) as client:
            raw = await client.call_tool(tool, args or {})
    except ToolError as e:
        return {'status': 'error', 'tool': tool, 'message': str(e)}
    except Exception as e:
        return {'status': 'error', 'tool': tool, 'message': f'{type(e).__name__}: {e}'}

    # FastMCP 3.x returns a CallToolResult object, not a list.
    content = raw.content if hasattr(raw, 'content') else (raw or [])
    if not content:
        return None
    item = content[0]
    text = item.text if hasattr(item, 'text') else str(item)
    try:
        return json.loads(text)
    except (json.JSONDecodeError, ValueError):
        return text


def run(tool: str, args: dict = None):
    """Synchronous wrapper — use in notebook cells."""
    result = asyncio.run(_call(tool, args))
    print(json.dumps(result, indent=2, default=str))
    return result


def skip_llm() -> bool:
    """Return True and print a notice when the LLM key is absent."""
    if not HAS_LLM_KEY:
        print('[SKIPPED] Set RADIANT_OLLAMA_API_KEY to run LLM cells.')
        return True
    return False


def assert_ok(result, field: str = None):
    """Raise AssertionError on error or missing field."""
    assert result is not None, 'Tool returned no result'
    if isinstance(result, dict):
        if result.get('status') == 'error':
            raise AssertionError(
                f'Tool error: {result.get("message", result)}\n'
                f'If this is a Redis connection error, see the note in the'
                f' Query section about fixing the mcp branch.'
            )
        if field:
            assert field in result, f'Expected field "{field}" missing: {result}'
    print('[OK]')


async def _wait_for_server(url: str, timeout: int = 120, interval: float = 1.0) -> bool:
    """Poll until the server accepts connections or timeout expires."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            async with _httpx.AsyncClient() as http:
                await http.get(
                    url, timeout=2.0,
                    headers={'Accept': 'application/json, text/event-stream'}
                )
                return True
        except (_httpx.ConnectError, _httpx.ConnectTimeout, _httpx.ReadTimeout):
            await asyncio.sleep(interval)
    return False


print('Helpers loaded  ✓')


Helpers loaded  ✓


## 5  Server startup

The LLM backend must use lazy initialization (no network calls in `__init__`) per the
spec. If the server does not reach `ready` within 30 s, the LLM backend is still
making an eager connection — fix it in the `mcp` branch before continuing.


In [5]:
import subprocess, time as _time

# ── Kill any process already holding the port ─────────────────────────────────
_port = int(os.environ['RADIANT_PORT'])
subprocess.run(['fuser', '-k', f'{_port}/tcp'], capture_output=True)
_time.sleep(1.0)

logging.basicConfig(stream=__import__('sys').stderr, level=logging.WARNING, force=True)

_server_ready = threading.Event()
_server_error  = None
_transport_used = None

def _run_server():
    global _server_error, _transport_used
    try:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        from radiant_rag_mcp.server import mcp
        print('Package      : radiant_rag_mcp  ✓')
        _server_ready.set()

        host = os.environ['RADIANT_HOST']
        port = int(os.environ['RADIANT_PORT'])

        # FastMCP changed the Streamable HTTP transport name across versions.
        # Try each in order; the first one that doesn't raise immediately wins.
        for _t in ['http', 'streamable-http']:
            try:
                _transport_used = _t
                mcp.run(transport=_t, host=host, port=port)
                return                # clean exit (server stopped normally)
            except Exception as _e:
                if 'Unknown transport' in str(_e) or 'unknown transport' in str(_e).lower():
                    continue          # this name isn't supported, try next
                raise                 # real error — stop trying
        raise RuntimeError("No supported HTTP transport found (tried: http, streamable-http)")

    except Exception as exc:
        _server_error = exc
        if not _server_ready.is_set():
            _server_ready.set()

_thread = threading.Thread(target=_run_server, name='radiant-mcp', daemon=True)
_thread.start()

if not _server_ready.wait(timeout=30):
    raise TimeoutError('Server thread did not signal ready within 30 s.')
if _server_error:
    raise _server_error

print(f'Waiting for server at {SERVER_URL} ...')
_deadline = time.time() + 90
while time.time() < _deadline:
    if _server_error:
        raise RuntimeError(f'Server thread failed: {_server_error}')
    if asyncio.run(_wait_for_server(SERVER_URL, timeout=5)):
        print(f'Server ready  ✓  →  {SERVER_URL}  (transport: {_transport_used})')
        break
    time.sleep(1)
else:
    raise TimeoutError(
        f'Server did not bind to {SERVER_URL} within 90 s.\n'
        f'Transport tried: {_transport_used}\n'
        f'Error (if any): {_server_error}'
    )


Package      : radiant_rag_mcp  ✓
Waiting for server at http://127.0.0.1:8765/mcp ...


╭──────────────────────────────────────────────────────────────────────────────╮                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │                  
                 │                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                                FastMCP 3.2.3                                 │                  
                 │                            https://gofastmcp.com                             │                  
                 │                                                                              │                  
                 │                  🖥  Server:      radiant-rag, 3.2.3                          │                  
                 │                  🚀 Deploy free: https://horizon.prefect.io                  │                  
                 │                                                                              │                  
                 ╰──────────────────────────────────────────────────────────────────────────────╯

2026-04-13 20:09:18,940 - radiant_rag_mcp.app - INFO - Initializing Radiant RAG...
2026-04-13 20:09:18,941 - radiant_rag_mcp.llm.client - INFO - Using new backend configuration system
2026-04-13 20:09:18,944 - radiant_rag_mcp.llm.backends.factory - INFO - Creating LLM backend: type=ollama
2026-04-13 20:09:18,945 - radiant_rag_mcp.llm.backends.llm_backends - INFO - Configured OpenAI-compatible LLM backend: model=gemma3:12b-cloud, base_url=https://ollama.com/v1
2026-04-13 20:09:18,945 - radiant_rag_mcp.llm.backends.factory - INFO - Creating embedding backend: type=local
2026-04-13 20:09:18,946 - radiant_rag_mcp.llm.backends.embedding_backends - INFO - Initializing sentence-transformers embedding backend: sentence-transformers/all-MiniLM-L12-v2
The following layers were not sharded: encoder.layer.*.intermediate.dense.weight, embeddings.LayerNorm.bias, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.output.dense.weight, encoder.layer.*.attention.self.value.bias, pooler.dense.bias,

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-04-13 20:09:25,310 - radiant_rag_mcp.utils.cache - INFO - Initialized EmbeddingCache with max_size=10000 (true LRU)
2026-04-13 20:09:25,311 - radiant_rag_mcp.llm.backends.embedding_backends - INFO - Loaded sentence-transformers embedding model: sentence-transformers/all-MiniLM-L12-v2 (dim=384, device=cuda)
2026-04-13 20:09:25,312 - radiant_rag_mcp.llm.backends.factory - INFO - Creating reranking backend: type=local
2026-04-13 20:09:25,312 - radiant_rag_mcp.llm.backends.reranking_backends - INFO - Initializing cross-encoder reranking backend: cross-encoder/ms-marco-MiniLM-L12-v2
The following layers were not sharded: classifier.weight, bert.embeddings.position_embeddings.weight, bert.encoder.layer.*.attention.output.LayerNorm.weight, classifier.bias, bert.encoder.layer.*.attention.self.query.weight, bert.encoder.layer.*.attention.output.LayerNorm.bias, bert.encoder.layer.*.attention.self.value.weight, bert.encoder.layer.*.intermediate.dense.weight, bert.embeddings.LayerNorm.weight,

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-04-13 20:09:28,316 - radiant_rag_mcp.llm.backends.reranking_backends - INFO - Loaded cross-encoder reranking model: cross-encoder/ms-marco-MiniLM-L12-v2 (device=cuda)
2026-04-13 20:09:28,319 - radiant_rag_mcp.storage.factory - INFO - Creating Chroma vector store
2026-04-13 20:09:28,457 - radiant_rag_mcp.storage.chroma_store - INFO - Initialized ChromaDB store at 'data/chroma_db' with collection 'radiant_docs'
2026-04-13 20:09:28,458 - radiant_rag_mcp.utils.conversation - INFO - ConversationStore: storage backend is 'chroma' — using in-memory storage
2026-04-13 20:09:28,470 - radiant_rag_mcp.utils.metrics_export - INFO - Prometheus metrics exporter initialized
2026-04-13 20:09:28,471 - PlanningAgent - INFO - [adab1387] [enabled=True] Initialized PlanningAgent
2026-04-13 20:09:28,472 - QueryDecompositionAgent - INFO - [627110e0] [enabled=True] Initialized QueryDecompositionAgent
2026-04-13 20:09:28,472 - QueryRewriteAgent - INFO - [1fb24471] [enabled=True] Initialized QueryRewriteAg

[04/13/26 20:09:28] INFO     Starting MCP server 'radiant-rag' with transport 'http' on            ]8;id=736990;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/mixins/transport.py\transport.py]8;;\:]8;id=136795;file:///usr/local/lib/python3.12/dist-packages/fastmcp/server/mixins/transport.py#299\299]8;;\
                             http://127.0.0.1:8765/mcp                                                             

INFO:     Started server process [24432]
INFO:     Waiting for application startup.
2026-04-13 20:09:28,561 - mcp.server.streamable_http_manager - INFO - StreamableHTTP session manager started
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8765 (Press CTRL+C to quit)
2026-04-13 20:09:29,382 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 13161e01788e4fa2b0f55db0bbe87ef3


INFO:     127.0.0.1:55442 - "GET /mcp HTTP/1.1" 400 Bad Request
Server ready  ✓  →  http://127.0.0.1:8765/mcp  (transport: http)


## 6  Connection verification


In [6]:
async def _list_tools():
    async with Client(SERVER_URL) as client:
        return await client.list_tools()

tools = asyncio.run(_list_tools())
print(f'{len(tools)} tool(s) registered:\n')
for t in sorted(tools, key=lambda x: x.name):
    desc = (t.description or '').splitlines()[0]
    print(f'  {t.name:<24}  {desc}')

EXPECTED = {
    'search_knowledge', 'query_knowledge', 'simple_query', 'start_conversation',
    'ingest_documents', 'ingest_url', 'get_index_stats', 'clear_index',
    'rebuild_bm25', 'set_ingest_mode',
}
registered = {t.name for t in tools}
missing    = EXPECTED - registered
extra      = registered - EXPECTED

print()
if missing:
    print(f'[WARNING] Missing tools : {sorted(missing)}')
if extra:
    print(f'[NOTE]    Extra tools   : {sorted(extra)}')
if not missing:
    print('All 10 expected tools registered  ✓')


2026-04-13 20:11:13,735 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: be8fdbb5077a46c7b3ae1b74479241df


INFO:     127.0.0.1:36208 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:13,743 - mcp.client.streamable_http - INFO - Received session ID: be8fdbb5077a46c7b3ae1b74479241df
2026-04-13 20:11:13,744 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:36222 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:36234 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:36236 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:13,759 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 20:11:13,768 - mcp.server.streamable_http - INFO - Terminating session: be8fdbb5077a46c7b3ae1b74479241df


INFO:     127.0.0.1:36252 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:13,771 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


10 tool(s) registered:

  clear_index               Delete all indexed documents from the knowledge base.
  get_index_stats           Return document counts and system health for the knowledge base.
  ingest_documents          Index local files or directories into the knowledge base.
  ingest_url                Index a URL or GitHub repository into the knowledge base.
  query_knowledge           Answer a question using the full agentic RAG pipeline.
  rebuild_bm25              Rebuild the BM25 sparse index from the current vector store contents.
  search_knowledge          Retrieve documents from the knowledge base without LLM generation.
  set_ingest_mode           Set the default ingestion storage mode for this server session.
  simple_query              Answer a question using a lightweight retrieval-and-synthesis pipeline.
  start_conversation        Create a new conversation ID for multi-turn query sessions.

All 10 expected tools registered  ✓


## 7  Pre-ingestion state


In [7]:
print('=== get_index_stats (before ingestion) ===')
r = run('get_index_stats')
assert_ok(r)


2026-04-13 20:11:16,995 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 5fb842588e074428a254e30c1491e31d


=== get_index_stats (before ingestion) ===
INFO:     127.0.0.1:37956 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:17,000 - mcp.client.streamable_http - INFO - Received session ID: 5fb842588e074428a254e30c1491e31d
2026-04-13 20:11:17,000 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:37970 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:37982 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:37996 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:17,015 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-13 20:11:17,021 - radiant_rag_mcp.storage.bm25_index - INFO - Creating new BM25 index


INFO:     127.0.0.1:38008 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:17,030 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 20:11:17,038 - mcp.server.streamable_http - INFO - Terminating session: 5fb842588e074428a254e30c1491e31d


INFO:     127.0.0.1:38010 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:17,040 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "health": {
    "redis": true,
    "vector_index": {
      "name": "radiant_docs",
      "backend": "chroma",
      "persist_directory": "./data/chroma_db",
      "document_count": 0,
      "distance_function": "cosine",
      "embedding_dimension": 384,
      "metadata": {
        "hnsw:space": "cosine"
      }
    },
    "bm25_index": {
      "document_count": 0,
      "unique_terms": 0,
      "avg_doc_length": 0.0,
      "k1": 1.5,
      "b": 0.75,
      "dirty": false,
      "needs_rebuild": true,
      "index_path": "data/bm25_index.json.gz",
      "storage_format": "json.gz (pending)"
    },
    "conversation": true
  },
  "metrics": {
    "run_count": 0,
    "average_latency_ms": null,
    "average_success_rate": 1.0,
    "history": []
  }
}
[OK]


In [8]:
# Verify set_ingest_mode: confirm default, switch to flat, restore.
print('=== set_ingest_mode: hierarchical=True ===')
r = run('set_ingest_mode', {'hierarchical': True})
assert_ok(r, 'hierarchical')
assert r['hierarchical'] is True

print('\n=== set_ingest_mode: hierarchical=False ===')
r = run('set_ingest_mode', {'hierarchical': False})
assert_ok(r, 'hierarchical')
assert r['hierarchical'] is False

print('\n=== set_ingest_mode: restore ===')
r = run('set_ingest_mode', {'hierarchical': True})
assert r['hierarchical'] is True
print('[OK] Restored to hierarchical.')


2026-04-13 20:11:21,996 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: e2cc9e9ad23846c5800aae3d5579574d


=== set_ingest_mode: hierarchical=True ===
INFO:     127.0.0.1:38022 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:22,001 - mcp.client.streamable_http - INFO - Received session ID: e2cc9e9ad23846c5800aae3d5579574d
2026-04-13 20:11:22,002 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:38024 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:38026 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:38032 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:22,015 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest


...

INFO:     127.0.0.1:38148 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:22,216 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 20:11:22,224 - mcp.server.streamable_http - INFO - Terminating session: 6e80b9dc83ca43849b41a5fdbc75f1da


INFO:     127.0.0.1:38150 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:22,227 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "hierarchical": true,
  "message": "Ingest mode set to 'hierarchical'.  All subsequent ingest_documents calls will use hierarchical storage by default."
}
[OK] Restored to hierarchical.


## 8  Document ingestion


In [9]:
TEST_DIR = Path('/tmp/radiant_test_docs')
TEST_DIR.mkdir(parents=True, exist_ok=True)

docs = {
    'rag_overview.md': textwrap.dedent("""
        # Retrieval-Augmented Generation

        Retrieval-Augmented Generation (RAG) combines dense vector search with
        large language models to answer questions grounded in a document corpus.

        ## Key components
        - **Embedding model** — converts text chunks into dense vectors.
        - **Vector store** — indexes vectors for approximate nearest-neighbour search.
        - **BM25 index** — keyword-based sparse retrieval for exact term matching.
        - **Hybrid fusion** — Reciprocal Rank Fusion (RRF) combines dense and sparse results.
        - **LLM** — synthesises a grounded answer from retrieved context.
    """).strip(),

    'storage_backends.md': textwrap.dedent("""
        # Storage Backends

        Radiant RAG supports three vector storage backends.

        ## ChromaDB
        Default backend. Embedded, no external services required.
        Best for development, testing, and small corpora.

        ## Redis Stack
        Production backend. Requires Redis Stack via Docker or native install.
        Supports HNSW indexing, binary quantization, and sub-millisecond latency.
        Docker: docker run -d --name redis-stack -p 6379:6379 redis/redis-stack-server

        ## PostgreSQL with pgvector
        Enterprise backend. Requires PostgreSQL with the pgvector extension.
        ACID-compliant; integrates with existing PostgreSQL infrastructure.
        Docker: docker run -d --name pgvector -p 5432:5432 pgvector/pgvector:pg16
    """).strip(),

    'performance.txt': textwrap.dedent("""
        Performance optimisations in Radiant RAG

        Intelligent caching: LRU cache for embeddings and query results.
        Embedding cache hit rate: 30-50 percent for typical workloads.
        Query cache hit rate: 20-40 percent.
        Repeated queries: up to 93 percent latency reduction on cache hits.

        Parallel execution: dense and BM25 retrieval run concurrently.
        Hybrid retrieval is approximately 50 percent faster than sequential.

        Binary quantization: 10-20x faster retrieval, 3.5x memory reduction.
        Accuracy retention with binary quantization: 95-96 percent.
    """).strip(),
}

for fname, content in docs.items():
    (TEST_DIR / fname).write_text(content)
    print(f'  created  {TEST_DIR / fname}')


  created  /tmp/radiant_test_docs/rag_overview.md
  created  /tmp/radiant_test_docs/storage_backends.md
  created  /tmp/radiant_test_docs/performance.txt


In [10]:
print('=== ingest_documents (hierarchical=True) ===')
r = run('ingest_documents', {'paths': [str(TEST_DIR)], 'hierarchical': True})
assert_ok(r)


2026-04-13 20:11:30,142 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 92cb99636aa24c6eac37df0783c92613


=== ingest_documents (hierarchical=True) ===
INFO:     127.0.0.1:47844 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:30,147 - mcp.client.streamable_http - INFO - Received session ID: 92cb99636aa24c6eac37df0783c92613
2026-04-13 20:11:30,148 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:47850 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:47860 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:47870 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:30,162 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-13 20:11:30,997 - radiant_rag_mcp.storage.bm25_index - INFO - Syncing BM25 index with store...
2026-04-13 20:11:31,005 - radiant_rag_mcp.storage.bm25_index - INFO - Saved BM25 index with 6 documents
2026-04-13 20:11:31,005 - radiant_rag_mcp.storage.bm25_index - INFO - Sync complete: 6 added, 0 removed


INFO:     127.0.0.1:47872 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:31,014 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 20:11:31,022 - mcp.server.streamable_http - INFO - Terminating session: 92cb99636aa24c6eac37df0783c92613


INFO:     127.0.0.1:47876 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:31,024 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "files_processed": 3,
  "files_failed": 0,
  "chunks_created": 3,
  "documents_stored": 9,
  "errors": []
}
[OK]


In [11]:
%%time
# Fetch a single page — no_crawl=True avoids following links.
print('=== ingest_url (no_crawl=True) ===')
r = run('ingest_url', {
    'url': 'https://raw.githubusercontent.com/dshipley71/radiant-rag/main/README.md',
    'no_crawl': True,
})
assert_ok(r)


2026-04-13 20:11:35,444 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 69a257f83199440e9442c8035d6294c8


=== ingest_url (no_crawl=True) ===
INFO:     127.0.0.1:41588 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:35,447 - mcp.client.streamable_http - INFO - Received session ID: 69a257f83199440e9442c8035d6294c8
2026-04-13 20:11:35,449 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:41592 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:41608 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:41618 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:35,462 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-13 20:11:35,465 - radiant_rag_mcp.ingestion.web_crawler - INFO - Starting crawl with 1 seed URLs, max_depth=0
2026-04-13 20:11:35,804 - radiant_rag_mcp.ingestion.web_crawler - INFO - Crawl complete: 1 pages crawled, 0 failed, 43317 bytes
2026-04-13 20:11:35,805 - radiant_rag_mcp.app - INFO - Crawled 1 pages, 0 failed, 43317 bytes
2026-04-13 20:11:36,090 - radiant_rag_mcp.storage.bm25_index - INFO - Syncing BM25 index with store...
2026-04-13 20:11:36,165 - radiant_rag_mcp.storage.bm25_index - INFO - Saved BM25 index with 76 documents
2026-04-13 20:11:36,166 - radiant_rag_mcp.storage.bm25_index - INFO - Sync complete: 70 added, 0 removed


INFO:     127.0.0.1:41622 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:36,175 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 20:11:36,182 - mcp.server.streamable_http - INFO - Terminating session: 69a257f83199440e9442c8035d6294c8


INFO:     127.0.0.1:41632 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:36,185 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "urls_crawled": 1,
  "files_processed": 1,
  "files_failed": 0,
  "chunks_created": 70,
  "documents_stored": 140,
  "github_repos": 0,
  "errors": []
}
[OK]
CPU times: user 481 ms, sys: 79.2 ms, total: 560 ms
Wall time: 797 ms


In [12]:
%%time
print('=== get_index_stats (after ingestion) ===')
r = run('get_index_stats')
assert_ok(r)
# Document count is nested: health.vector_index.document_count
total = (
    r.get('health', {}).get('vector_index', {}).get('document_count', 0)
    if isinstance(r, dict) else 0
)
assert total > 0, f'Expected documents > 0, got {total}'
print(f'[OK] {total} document(s) indexed.')


2026-04-13 20:11:40,534 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 0a20acde25c1409bb598cad5e2e934b5


=== get_index_stats (after ingestion) ===
INFO:     127.0.0.1:41642 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:40,539 - mcp.client.streamable_http - INFO - Received session ID: 0a20acde25c1409bb598cad5e2e934b5
2026-04-13 20:11:40,539 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:41650 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:41654 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:41670 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:40,552 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest


INFO:     127.0.0.1:41674 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:40,564 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 20:11:40,571 - mcp.server.streamable_http - INFO - Terminating session: 0a20acde25c1409bb598cad5e2e934b5


INFO:     127.0.0.1:41676 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:40,574 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "health": {
    "redis": true,
    "vector_index": {
      "name": "radiant_docs",
      "backend": "chroma",
      "persist_directory": "./data/chroma_db",
      "document_count": 149,
      "distance_function": "cosine",
      "embedding_dimension": 384,
      "metadata": {
        "hnsw:space": "cosine"
      }
    },
    "bm25_index": {
      "document_count": 76,
      "unique_terms": 952,
      "avg_doc_length": 51.71052631578947,
      "k1": 1.5,
      "b": 0.75,
      "dirty": false,
      "needs_rebuild": true,
      "index_path": "data/bm25_index.json.gz",
      "storage_format": "json.gz"
    },
    "conversation": true
  },
  "metrics": {
    "run_count": 0,
    "average_latency_ms": null,
    "average_success_rate": 1.0,
    "history": []
  }
}
[OK]
[OK] 149 document(s) indexed.
CPU times: user 92.9 ms, sys: 6.63 ms, total: 99.5 ms
Wall time: 95.1 ms


## 9  Search  *(no LLM required)*


In [13]:
%%time
print('=== search_knowledge: mode=hybrid ===')
r = run('search_knowledge', {
    'query': 'hybrid retrieval BM25 vector search RRF',
    'mode': 'hybrid',
    'top_k': 3,
})
assert_ok(r)


2026-04-13 20:11:44,902 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 81e49cbf3a324822ac8bf5cfc648d62e


=== search_knowledge: mode=hybrid ===
INFO:     127.0.0.1:51662 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:44,914 - mcp.client.streamable_http - INFO - Received session ID: 81e49cbf3a324822ac8bf5cfc648d62e
2026-04-13 20:11:44,915 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:51678 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:51692 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:51704 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:44,928 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-13 20:11:44,930 - DenseRetrievalAgent - INFO - [0d3704a1] [enabled=True] Initialized DenseRetrievalAgent
2026-04-13 20:11:45,048 - DenseRetrievalAgent - INFO - [fe8ffce9] [query_length=39 num_results=3 top_k=3] Dense retrieval completed
2026-04-13 20:11:45,049 - DenseRetrievalAgent - INFO - [fe8ffce9] [duration_ms=118.16 status=success] Execution completed
2026-04-13 20:11:45,050 - BM25RetrievalAgent - INFO - [c36210f7] [enabled=True] Initialized BM25RetrievalAgent
2026-04-13 20:11:45,056 - BM25RetrievalAgent - INFO - [b4cb746a] [query_length=39 num_results=3 top_k=3] BM25 retrieval completed
2026-04-13 20:11:45,057 - BM25RetrievalAgent - INFO - [b4cb746a] [duration_ms=6.72 status=success] Execution completed
2026-04-13 20:11:45,057 - RRFAgent - INFO - [d6f74448] [enabled=True] Initialized RRFAgent
2026-04-13 20:11:45,058 - RRFAgent - INFO - [dfb6c20b] [num_runs=2 total_docs

INFO:     127.0.0.1:51708 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:45,068 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 20:11:45,077 - mcp.server.streamable_http - INFO - Terminating session: 81e49cbf3a324822ac8bf5cfc648d62e


INFO:     127.0.0.1:51718 - "DELETE /mcp HTTP/1.1" 200 OK
{
  "query": "hybrid retrieval BM25 vector search RRF",
  "mode": "hybrid",
  "results": [
    {
      "doc_id": "2b55bb2f254c42b5f4b21f2037ff4635eace49c3e0051491a0602e007d1c7560",
      "content": "# Retrieval-Augmented Generation Retrieval-Augmented Generation (RAG) combines dense vector search with large language models to answer questions grounded in a document corpus. ## Key components - **Embedding model** \u2014 converts text chunks into dense vectors. - **Vector store** \u2014 indexes vectors for approximate nearest-neighbour search. - **BM25 index** \u2014 keyword-based sparse retrieval for exact term matching. - **Hybrid fusion** \u2014 Reciprocal Rank Fusion (RRF) combines dense and sparse results. - **LLM** \u2014",
      "meta": {
        "has_embedding": true,
        "doc_level": "child",
        "cleaning": {
          "enabled": true,
          "bullets": false,
          "extra_whitespace": true,
          "das

In [14]:
%%time
print('=== search_knowledge: mode=dense ===')
r = run('search_knowledge', {
    'query': 'storage backend production deployment comparison',
    'mode': 'dense',
    'top_k': 3,
})
assert_ok(r)


2026-04-13 20:11:50,234 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: c4dcdd0b38bc434da984cad30b17d594


=== search_knowledge: mode=dense ===
INFO:     127.0.0.1:51730 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:50,239 - mcp.client.streamable_http - INFO - Received session ID: c4dcdd0b38bc434da984cad30b17d594
2026-04-13 20:11:50,240 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:51746 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:51760 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:51768 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:50,252 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-13 20:11:50,254 - DenseRetrievalAgent - INFO - [b7a34a39] [enabled=True] Initialized DenseRetrievalAgent
2026-04-13 20:11:50,271 - DenseRetrievalAgent - INFO - [fe5caabc] [query_length=48 num_results=3 top_k=3] Dense retrieval completed
2026-04-13 20:11:50,272 - DenseRetrievalAgent - INFO - [fe5caabc] [duration_ms=17.02 status=success] Execution completed


INFO:     127.0.0.1:51774 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:50,280 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 20:11:50,288 - mcp.server.streamable_http - INFO - Terminating session: c4dcdd0b38bc434da984cad30b17d594


INFO:     127.0.0.1:51784 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:50,290 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "query": "storage backend production deployment comparison",
  "mode": "dense",
  "results": [
    {
      "doc_id": "6eaabd8c6dbf957acd2aa4b32647e515bc21fc2ba6cbb6c2275a17aba4098434",
      "content": "# Storage Backends Radiant RAG supports three vector storage backends. ## ChromaDB Default backend. Embedded, no external services required. Best for development, testing, and small corpora. ## Redis Stack Production backend. Requires Redis Stack via Docker or native install. Supports HNSW indexing, binary quantization, and sub-millisecond latency. Docker: docker run -d --name redis-stack -p 6379:6379 redis/redis-stack-server ## PostgreSQL with pgvector Enterprise backend. Requires PostgreSQL with the",
      "meta": {
        "has_embedding": true,
        "cleaning": {
          "enabled": true,
          "bullets": false,
          "extra_whitespace": true,
          "dashes": false,
          "trailing_punctuation": false,
          "lowercase": false
        },
        "doc_lev

In [15]:
%%time
print('=== search_knowledge: mode=bm25 ===')
r = run('search_knowledge', {
    'query': 'binary quantization latency memory reduction',
    'mode': 'bm25',
    'top_k': 3,
})
assert_ok(r)


2026-04-13 20:11:54,723 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 418599dadb92435b86cadbf819c1a8e0


=== search_knowledge: mode=bm25 ===
INFO:     127.0.0.1:46828 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:54,727 - mcp.client.streamable_http - INFO - Received session ID: 418599dadb92435b86cadbf819c1a8e0
2026-04-13 20:11:54,728 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:46838 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:46842 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:46850 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:54,741 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-13 20:11:54,743 - BM25RetrievalAgent - INFO - [a224fcee] [enabled=True] Initialized BM25RetrievalAgent
2026-04-13 20:11:54,746 - BM25RetrievalAgent - INFO - [1d109ac1] [query_length=44 num_results=3 top_k=3] BM25 retrieval completed
2026-04-13 20:11:54,747 - BM25RetrievalAgent - INFO - [1d109ac1] [duration_ms=3.48 status=success] Execution completed


INFO:     127.0.0.1:46858 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:54,755 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 20:11:54,762 - mcp.server.streamable_http - INFO - Terminating session: 418599dadb92435b86cadbf819c1a8e0


INFO:     127.0.0.1:46862 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-13 20:11:54,765 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "query": "binary quantization latency memory reduction",
  "mode": "bm25",
  "results": [
    {
      "doc_id": "4c6caa7250787d3d02e2c713b95226f11cc3a2f28177ea9066d53f49b05e23b7",
      "content": "memory reduction. Accuracy retention with binary quantization: 95-96 percent.",
      "meta": {
        "parent_id": "94877d7ea8b53f376d1919bca8bb95e02e9e83850c3e145d102df957ed6a99ee",
        "child_total": 2,
        "has_embedding": true,
        "doc_level": "child",
        "child_index": 1,
        "cleaning": {
          "enabled": true,
          "bullets": false,
          "extra_whitespace": true,
          "dashes": false,
          "trailing_punctuation": false,
          "lowercase": false
        },
        "source_path": "/tmp/radiant_test_docs/performance.txt"
      },
      "score": 11.4283488746968
    },
    {
      "doc_id": "a313774c6cfbb31ea2e67a330c468e3592959445025abc151d9cf87ed778adb8",
      "content": "Performance optimisations in Radiant RAG Intelligent cachin

## 10  Query  `[LLM]`

In [16]:
%%time
if not skip_llm():
    print('=== simple_query ===')
    r = run('simple_query', {
        'question': 'What storage backend is best for production use?',
        'top_k': 5,
    })
    assert_ok(r, 'answer')


2026-04-13 20:12:01,193 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 1d6603d1fb814053b6c2b394e4b0f146


=== simple_query ===
INFO:     127.0.0.1:46876 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:12:01,197 - mcp.client.streamable_http - INFO - Received session ID: 1d6603d1fb814053b6c2b394e4b0f146
2026-04-13 20:12:01,199 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:46892 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:46900 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:46914 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:12:01,212 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-13 20:12:01,282 - radiant_rag_mcp.llm.backends.llm_backends - INFO - Initialized OpenAI-compatible LLM backend: model=gemma3:12b-cloud, base_url=https://ollama.com/v1


INFO:     127.0.0.1:46924 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:12:02,601 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 20:12:02,609 - mcp.server.streamable_http - INFO - Terminating session: 1d6603d1fb814053b6c2b394e4b0f146


INFO:     127.0.0.1:46938 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-13 20:12:02,612 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "question": "What storage backend is best for production use?",
  "answer": "Redis Stack."
}
[OK]
CPU times: user 560 ms, sys: 62.5 ms, total: 623 ms
Wall time: 1.47 s


In [17]:
%%time
if not skip_llm():
    print('=== query_knowledge (single-turn) ===')
    r = run('query_knowledge', {
        'question': 'What are the performance trade-offs between ChromaDB and Redis Stack?',
        'mode': 'hybrid',
        'conversation_id': None,
    })
    assert_ok(r, 'answer')


2026-04-13 20:12:20,159 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 8eeba833678b4dbf8d1301b4946f2a1a


=== query_knowledge (single-turn) ===
INFO:     127.0.0.1:58990 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:12:20,165 - mcp.client.streamable_http - INFO - Received session ID: 8eeba833678b4dbf8d1301b4946f2a1a
2026-04-13 20:12:20,166 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:58998 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:59000 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:59014 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:12:20,179 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-13 20:12:20,182 - radiant_rag_mcp.orchestrator - INFO - [d437bf6a-3093-41df-847c-1d5c8c8d7533] Starting agentic pipeline for query: What are the performance trade-offs between ChromaDB and Redis Stack?...
2026-04-13 20:12:20,182 - radiant_rag_mcp.orchestrator - INFO - [d437bf6a-3093-41df-847c-1d5c8c8d7533] Retrieval mode: hybrid, fast_path: False
2026-04-13 20:12:21,452 - QueryRewriteAgent - INFO - [d437bf6a-3093-41df-847c-1d5c8c8d7533] [original=What are the performance trade-offs between Chroma rewritten=Compare ChromaDB and Redis Stack performance, cons] Query rewritten
2026-04-13 20:12:21,453 - QueryRewriteAgent - INFO - [d437bf6a-3093-41df-847c-1d5c8c8d7533] [duration_ms=1269.74 status=success] Execution completed
2026-04-13 20:12:23,426 - QueryExpansionAgent - INFO - [d437bf6a-3093-41df-847c-1d5c8c8d7533] [original=Compare ChromaDB and Redis Stack performance, cons num

INFO:     127.0.0.1:35326 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:12:38,114 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 20:12:38,122 - mcp.server.streamable_http - INFO - Terminating session: 8eeba833678b4dbf8d1301b4946f2a1a


INFO:     127.0.0.1:35328 - "DELETE /mcp HTTP/1.1" 200 OK
{
  "run_id": "d437bf6a-3093-41df-847c-1d5c8c8d7533",
  "answer": "Here's a breakdown of the performance trade-offs between ChromaDB and Redis Stack, based on the provided documents:\n\n*   **ChromaDB:** According to [DOC 1], ChromaDB is the default backend and is \"best for development, testing, and small corpora.\" It is an embedded solution, meaning it doesn't require external services.\n*   **Redis Stack:** [DOC 1] states that Redis Stack is a \"production backend\" requiring Docker or a native installation. It supports HNSW indexing, binary quantization, and \"sub-millisecond latency.\" [DOC 2] & [DOC 7] highlight that Redis Stack is part of a system offering \"60-93% faster\" performance through intelligent caching, parallel execution, and batching. [DOC 6] also mentions the benefits of binary quantization with Redis Stack: 10-20x faster retrieval and 3.5x less memory usage.\n\n\n\nIn essence, ChromaDB is suitable for smal

## 11  Multi-turn conversation  `[LLM]`


In [18]:
%%time
# start_conversation has no LLM dependency — always runs.
print('=== start_conversation ===')
conv = run('start_conversation', {})
assert_ok(conv, 'conversation_id')
conversation_id = conv['conversation_id']
print(f'\nconversation_id: {conversation_id}')


2026-04-13 20:13:42,773 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: cda10a74044a45c484aafe18d027f3d6


=== start_conversation ===
INFO:     127.0.0.1:42340 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:13:42,778 - mcp.client.streamable_http - INFO - Received session ID: cda10a74044a45c484aafe18d027f3d6
2026-04-13 20:13:42,779 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:42356 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:42358 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:42366 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:13:42,792 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest


INFO:     127.0.0.1:42372 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:13:42,800 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 20:13:42,808 - mcp.server.streamable_http - INFO - Terminating session: cda10a74044a45c484aafe18d027f3d6


INFO:     127.0.0.1:42388 - "DELETE /mcp HTTP/1.1" 200 OK
{
  "conversation_id": "2b4ee349-bf46-4c88-a090-0cf4e906870a"
}
[OK]

conversation_id: 2b4ee349-bf46-4c88-a090-0cf4e906870a
CPU times: user 86.6 ms, sys: 8.3 ms, total: 94.9 ms
Wall time: 90.9 ms


In [19]:
%%time
if not skip_llm():
    print(f'=== query_knowledge: turn 1  (conv={conversation_id}) ===')
    r = run('query_knowledge', {
        'question': 'Which backend is best for a production deployment?',
        'mode': 'hybrid',
        'conversation_id': conversation_id,
    })
    assert_ok(r, 'answer')


2026-04-13 20:13:46,203 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 09691442aa71410cb58cb90a53883db3


=== query_knowledge: turn 1  (conv=2b4ee349-bf46-4c88-a090-0cf4e906870a) ===
INFO:     127.0.0.1:36746 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:13:46,212 - mcp.client.streamable_http - INFO - Received session ID: 09691442aa71410cb58cb90a53883db3
2026-04-13 20:13:46,213 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:36754 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:36756 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:36758 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:13:46,226 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-13 20:13:46,228 - radiant_rag_mcp.orchestrator - INFO - [fe6fb417-c3b0-412b-9822-c3932cf06456] Starting agentic pipeline for query: Which backend is best for a production deployment?...
2026-04-13 20:13:46,229 - radiant_rag_mcp.orchestrator - INFO - [fe6fb417-c3b0-412b-9822-c3932cf06456] Retrieval mode: hybrid, fast_path: True
2026-04-13 20:13:47,429 - QueryRewriteAgent - INFO - [fe6fb417-c3b0-412b-9822-c3932cf06456] [original=Which backend is best for a production deployment? rewritten=Compare backend technologies suitable for producti] Query rewritten
2026-04-13 20:13:47,430 - QueryRewriteAgent - INFO - [fe6fb417-c3b0-412b-9822-c3932cf06456] [duration_ms=1200.75 status=success] Execution completed
2026-04-13 20:13:47,452 - RRFAgent - INFO - [fe6fb417-c3b0-412b-9822-c3932cf06456] [num_runs=2 total_docs=9 fused_results=9] RRF fusion completed
2026-04-13 20:13:47,452 - RRFAge

INFO:     127.0.0.1:53648 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:13:56,597 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 20:13:56,604 - mcp.server.streamable_http - INFO - Terminating session: 09691442aa71410cb58cb90a53883db3


INFO:     127.0.0.1:53662 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-13 20:13:56,607 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "run_id": "fe6fb417-c3b0-412b-9822-c3932cf06456",
  "answer": "According to [DOC 1], the production backend for Radiant RAG is Redis Stack, which supports HNSW indexing, binary quantization, and sub-millisecond latency. [DOC 2] also states that Redis (the default) is suitable for production and low-latency use cases.\n\n## Sources\n\n[1] **Source 1**\n    File: /tmp/radiant_test_docs/storage_backends.md\n[2] **Source 2**\n    URL: https://raw.githubusercontent.com/dshipley71/radiant-rag/main/README.md\n[3] **Source 3**\n    URL: https://raw.githubusercontent.com/dshipley71/radiant-rag/main/README.md\n[4] **Source 4**\n    URL: https://raw.githubusercontent.com/dshipley71/radiant-rag/main/README.md\n[5] **Source 5**\n    URL: https://raw.githubusercontent.com/dshipley71/radiant-rag/main/README.md\n[6] **Source 6**\n    URL: https://raw.githubusercontent.com/dshipley71/radiant-rag/main/README.md\n[7] **Source 7**\n    URL: https://raw.githubusercontent.com/dshipley71/radiant-rag/main

In [20]:
%%time
if not skip_llm():
    print(f'=== query_knowledge: turn 2 follow-up  (conv={conversation_id}) ===')
    r = run('query_knowledge', {
        'question': 'What Docker command do I need to run for that backend?',
        'mode': 'hybrid',
        'conversation_id': conversation_id,
    })
    assert_ok(r, 'answer')


2026-04-13 20:14:00,526 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 71221be2b47844d2ae5f86d5b30be8ad


=== query_knowledge: turn 2 follow-up  (conv=2b4ee349-bf46-4c88-a090-0cf4e906870a) ===
INFO:     127.0.0.1:53674 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:14:00,531 - mcp.client.streamable_http - INFO - Received session ID: 71221be2b47844d2ae5f86d5b30be8ad
2026-04-13 20:14:00,532 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:53676 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:53688 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:53698 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:14:00,546 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-13 20:14:00,547 - radiant_rag_mcp.orchestrator - INFO - [83f8acdf-d997-48d5-9484-64b50c6d32d8] Starting agentic pipeline for query: What Docker command do I need to run for that backend?...
2026-04-13 20:14:00,548 - radiant_rag_mcp.orchestrator - INFO - [83f8acdf-d997-48d5-9484-64b50c6d32d8] Retrieval mode: hybrid, fast_path: False
2026-04-13 20:14:02,278 - QueryRewriteAgent - INFO - [83f8acdf-d997-48d5-9484-64b50c6d32d8] [original=What Docker command do I need to run for that back rewritten=What Docker command is required to start or manage] Query rewritten
2026-04-13 20:14:02,279 - QueryRewriteAgent - INFO - [83f8acdf-d997-48d5-9484-64b50c6d32d8] [duration_ms=1729.98 status=success] Execution completed
2026-04-13 20:14:03,592 - QueryExpansionAgent - INFO - [83f8acdf-d997-48d5-9484-64b50c6d32d8] [original=What Docker command is required to start or manage num_expansions=8] 

INFO:     127.0.0.1:55222 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:14:12,482 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 20:14:12,490 - mcp.server.streamable_http - INFO - Terminating session: 71221be2b47844d2ae5f86d5b30be8ad


INFO:     127.0.0.1:55228 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-13 20:14:12,493 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "run_id": "83f8acdf-d997-48d5-9484-64b50c6d32d8",
  "answer": "According to [DOC 1], the Docker command for the Redis Stack backend (the production backend) is: `docker run -d --name redis-stack -p 6379:6379 redis/redis-stack-server`. [DOC 2] and [DOC 3] also list this command.\n\n## Sources\n\n[1] **Source 1**\n    File: /tmp/radiant_test_docs/storage_backends.md\n[2] **Source 2**\n    URL: https://raw.githubusercontent.com/dshipley71/radiant-rag/main/README.md\n[3] **Source 3**\n    URL: https://raw.githubusercontent.com/dshipley71/radiant-rag/main/README.md\n[4] **Source 4**\n    URL: https://raw.githubusercontent.com/dshipley71/radiant-rag/main/README.md\n[5] **Source 5**\n    URL: https://raw.githubusercontent.com/dshipley71/radiant-rag/main/README.md\n[6] **Source 6**\n    URL: https://raw.githubusercontent.com/dshipley71/radiant-rag/main/README.md\n[7] **Source 7**\n    URL: https://raw.githubusercontent.com/dshipley71/radiant-rag/main/README.md\n[8] **Source 8**\n    URL: h

## 12  Maintenance


In [21]:
print('=== rebuild_bm25 ===')
r = run('rebuild_bm25', {})
assert_ok(r)

print('\n=== get_index_stats (after rebuild) ===')
run('get_index_stats')


2026-04-13 20:14:15,484 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: 661ec91905c14d3bb3a84f41a50e178a


=== rebuild_bm25 ===
INFO:     127.0.0.1:33046 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:14:15,489 - mcp.client.streamable_http - INFO - Received session ID: 661ec91905c14d3bb3a84f41a50e178a
2026-04-13 20:14:15,491 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:33054 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:33062 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:33064 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:14:15,503 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-13 20:14:15,504 - radiant_rag_mcp.storage.bm25_index - INFO - Building BM25 index from store (max 100000 documents)...
2026-04-13 20:14:15,580 - radiant_rag_mcp.storage.bm25_index - INFO - Saved BM25 index with 76 documents
2026-04-13 20:14:15,580 - radiant_rag_mcp.storage.bm25_index - INFO - Built BM25 index with 76 documents


...

2026-04-13 20:14:15,675 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest


INFO:     127.0.0.1:33104 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:14:15,693 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 20:14:15,702 - mcp.server.streamable_http - INFO - Terminating session: 900db22ae0b04a3babec62802131146d


INFO:     127.0.0.1:33118 - "DELETE /mcp HTTP/1.1" 200 OK
{
  "health": {
    "redis": true,
    "vector_index": {
      "name": "radiant_docs",
      "backend": "chroma",
      "persist_directory": "./data/chroma_db",
      "document_count": 149,
      "distance_function": "cosine",
      "embedding_dimension": 384,
      "metadata": {
        "hnsw:space": "cosine"
      }
    },
    "bm25_index": {
      "document_count": 76,
      "unique_terms": 952,
      "avg_doc_length": 51.71052631578949,
      "k1": 1.5,
      "b": 0.75,
      "dirty": false,
      "needs_rebuild": true,
      "index_path": "data/bm25_index.json.gz",
      "storage_format": "json.gz"
    },
    "conversation": true
  },
  "metrics": {
    "run_count": 3,
    "average_latency_ms": 13401.392936706543,
    "average_success_rate": 1.0,
    "history": [
      {
        "run_id": "d437bf6a-3093-41df-847c-1d5c8c8d7533",
        "started_at": 1776111140.182148,
        "ended_at": 1776111158.1039433,
        "total_l

{'health': {'redis': True,
  'vector_index': {'name': 'radiant_docs',
   'backend': 'chroma',
   'persist_directory': './data/chroma_db',
   'document_count': 149,
   'distance_function': 'cosine',
   'embedding_dimension': 384,
   'metadata': {'hnsw:space': 'cosine'}},
  'bm25_index': {'document_count': 76,
   'unique_terms': 952,
   'avg_doc_length': 51.71052631578949,
   'k1': 1.5,
   'b': 0.75,
   'dirty': False,
   'needs_rebuild': True,
   'index_path': 'data/bm25_index.json.gz',
   'storage_format': 'json.gz'},
  'conversation': True},
 'metrics': {'run_count': 3,
  'average_latency_ms': 13401.392936706543,
  'average_success_rate': 1.0,
  'history': [{'run_id': 'd437bf6a-3093-41df-847c-1d5c8c8d7533',
    'started_at': 1776111140.182148,
    'ended_at': 1776111158.1039433,
    'total_latency_ms': 17921.79536819458,
    'success_rate': 1.0,
    'steps': [{'name': 'QueryRewriteAgent',
      'started_at': 1776111140.183374,
      'ended_at': 1776111141.4537592,
      'latency_ms': 

In [22]:
# Dry run — confirm=False must return status=cancelled, no data deleted.
print('=== clear_index: dry run (confirm=False) ===')
r = run('clear_index', {'confirm': False})
assert isinstance(r, dict) and r.get('cleared') is False, \
    f'Expected cleared=False, got: {r}'
print('[OK] Cancellation returned as expected — no data deleted.')


2026-04-13 20:14:19,132 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: fc3333d5af62441e89f288fd7959ea50


=== clear_index: dry run (confirm=False) ===
INFO:     127.0.0.1:33130 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:14:19,143 - mcp.client.streamable_http - INFO - Received session ID: fc3333d5af62441e89f288fd7959ea50
2026-04-13 20:14:19,145 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:33142 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:33144 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:33160 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:14:19,157 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest


INFO:     127.0.0.1:33174 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:14:19,166 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 20:14:19,174 - mcp.server.streamable_http - INFO - Terminating session: fc3333d5af62441e89f288fd7959ea50


INFO:     127.0.0.1:33182 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-13 20:14:19,177 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "cleared": false,
  "message": "Index was NOT cleared.  Pass confirm=True to proceed.  This operation deletes all indexed documents and cannot be undone."
}
[OK] Cancellation returned as expected — no data deleted.


In [23]:
# WARNING: deletes all indexed documents.
# Comment this cell out to keep the test corpus.
print('=== clear_index: confirmed ===')
r = run('clear_index', {'confirm': True})
assert_ok(r)

print('\n=== get_index_stats (after clear) ===')
r = run('get_index_stats')
total = (
    r.get('health', {}).get('vector_index', {}).get('document_count', -1)
    if isinstance(r, dict) else -1
)
assert total == 0, f'Expected 0 documents after clear, got {total}'
print('[OK] Index empty.')


2026-04-13 20:14:21,495 - mcp.server.streamable_http_manager - INFO - Created new transport with session ID: e53c21ffb23140b89988ece0299614c2


=== clear_index: confirmed ===
INFO:     127.0.0.1:33190 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:14:21,499 - mcp.client.streamable_http - INFO - Received session ID: e53c21ffb23140b89988ece0299614c2
2026-04-13 20:14:21,500 - mcp.client.streamable_http - INFO - Negotiated protocol version: 2025-11-25


INFO:     127.0.0.1:33202 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:33214 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:33224 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:14:21,513 - mcp.server.lowlevel.server - INFO - Processing request of type CallToolRequest
2026-04-13 20:14:21,551 - radiant_rag_mcp.storage.chroma_store - INFO - Dropped Chroma collection 'radiant_docs'
2026-04-13 20:14:21,551 - radiant_rag_mcp.app - ERROR - Failed to clear index: 'ChromaVectorStore' object has no attribute '_ensure_index'


...

INFO:     127.0.0.1:33292 - "POST /mcp HTTP/1.1" 200 OK


2026-04-13 20:14:21,656 - mcp.server.lowlevel.server - INFO - Processing request of type ListToolsRequest
2026-04-13 20:14:21,662 - mcp.server.streamable_http - INFO - Terminating session: dd5879f6ba324e20b800277263ccca73


INFO:     127.0.0.1:33306 - "DELETE /mcp HTTP/1.1" 200 OK


2026-04-13 20:14:21,665 - mcp.client.streamable_http - INFO - GET stream disconnected, reconnecting in 1000ms...


{
  "health": {
    "redis": true,
    "vector_index": {
      "name": "radiant_docs",
      "backend": "chroma",
      "persist_directory": "./data/chroma_db",
      "document_count": 0,
      "distance_function": "cosine",
      "embedding_dimension": 384,
      "metadata": {
        "hnsw:space": "cosine"
      }
    },
    "bm25_index": {
      "document_count": 76,
      "unique_terms": 952,
      "avg_doc_length": 51.71052631578949,
      "k1": 1.5,
      "b": 0.75,
      "dirty": false,
      "needs_rebuild": true,
      "index_path": "data/bm25_index.json.gz",
      "storage_format": "json.gz"
    },
    "conversation": true
  },
  "metrics": {
    "run_count": 3,
    "average_latency_ms": 13401.392936706543,
    "average_success_rate": 1.0,
    "history": [
      {
        "run_id": "d437bf6a-3093-41df-847c-1d5c8c8d7533",
        "started_at": 1776111140.182148,
        "ended_at": 1776111158.1039433,
        "total_latency_ms": 17921.79536819458,
        "success_rate": 1.0,


## 13  Summary


In [24]:
print('=' * 55)
print('  Test run complete')
print('=' * 55)
print(f'  Branch        : {ACTIVE_BRANCH}')
print(f'  Server URL    : {SERVER_URL}')
print(f'  Backend       : {os.environ["RADIANT_STORAGE_BACKEND"]}')
print(f'  LLM cells ran : {HAS_LLM_KEY}')
print(f'  Test docs     : {TEST_DIR}')
print()
print('The server thread is daemonised and stops when the runtime ends.')
print('Re-run the server startup cell to restart it in the same session.')


  Test run complete
  Branch        : mcp
  Server URL    : http://127.0.0.1:8765/mcp
  Backend       : chroma
  LLM cells ran : True
  Test docs     : /tmp/radiant_test_docs

The server thread is daemonised and stops when the runtime ends.
Re-run the server startup cell to restart it in the same session.
